# Лабораторная работа по NLP №2

Выполнили студенты:
- Кудасов Максим, 21ПМИ-2
- Красильников Николай, 21ПМИ-1
- Ерёменко Даниил, 21ПМИ-1

## Подготовка

In [1]:
import os
import re
import logging
import math
from datetime import datetime

from transformers import (
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    AutoTokenizer,
    TrainerCallback,
)
from datasets import Dataset
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

os.environ["TOKENIZERS_PARALLELISM"] = "true"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.FileHandler("training.log"), logging.StreamHandler()],
)
logger = logging.getLogger(__name__)

In [2]:
class TextDataset(Dataset):
    def __init__(self, text, chunk_len=200, stride=50):
        self.text = text
        self.chunk_len = chunk_len
        self.stride = stride
        self.unique_chars = sorted(set(text))
        self.char_to_idx = {c: i for i, c in enumerate(self.unique_chars)}
        self.idx_to_char = {i: c for i, c in enumerate(self.unique_chars)}
        self.data = self._process_text()

    def __len__(self):
        return len(self.data)

    def _process_text(self):
        sequences = []
        for i in range(0, len(self.text) - self.chunk_len, self.stride):
            chunk = self.text[i:i+self.chunk_len+1]
            sequences.append(chunk)
        return sequences

    def __getitem__(self, idx):
        chunk = self.data[idx]
        input_seq = [self.char_to_idx[c] for c in chunk[:-1]]
        target_seq = [self.char_to_idx[c] for c in chunk[1:]]
        return torch.LongTensor(input_seq), torch.LongTensor(target_seq)

    @property
    def vocab_size(self):
        return len(self.unique_chars)

### Загрузка датасета с РИА Новости

### RNN

In [3]:
class CharRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, model="gru", n_layers=1, dropout=0.2):
        super().__init__()
        self.model_type = model.lower()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.n_layers = n_layers

        self.encoder = nn.Embedding(input_size, hidden_size)
        rnn_class = nn.GRU if self.model_type == "gru" else nn.LSTM
        self.rnn = rnn_class(
            hidden_size, hidden_size, n_layers,
            dropout=dropout if n_layers > 1 else 0,
            batch_first=True
        )
        self.dropout = nn.Dropout(dropout)
        self.decoder = nn.Linear(hidden_size, output_size)

    def forward(self, input, hidden=None):
        batch_size = input.size(0)
        encoded = self.encoder(input)
        output, hidden = self.rnn(encoded, hidden)
        output = self.dropout(output)
        decoded = self.decoder(output.contiguous().view(-1, self.hidden_size))
        return decoded.view(batch_size, -1, self.output_size), hidden

    def init_hidden(self, batch_size, device):
        if self.model_type == "lstm":
            return (
                torch.zeros(self.n_layers, batch_size, self.hidden_size).to(device),
                torch.zeros(self.n_layers, batch_size, self.hidden_size).to(device)
            )
        else:
            return torch.zeros(self.n_layers, batch_size, self.hidden_size).to(device)

In [4]:
def generate_sample(model, dataset, device, prompt="The", max_length=500, temperature=1.0, top_k=10, top_p=0.9):
    model.eval()
    generated = []
    input_seq = torch.LongTensor([dataset.char_to_idx[c] for c in prompt]).unsqueeze(0).to(device)
    hidden = model.init_hidden(1, device)

    with torch.inference_mode():
        if len(prompt) > 0:
            _, hidden = model(input_seq, hidden)

        input_seq = input_seq[:, -1].unsqueeze(1)

        for _ in range(max_length):
            outputs, hidden = model(input_seq, hidden)
            logits = outputs[:, -1, :] / temperature

            if top_k > 0:
                logits = _top_k_filter(logits, top_k)
            if top_p > 0.0:
                logits = _top_p_filter(logits, top_p)

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            generated.append(next_token.item())
            input_seq = next_token

    generated_str = prompt + ''.join([dataset.idx_to_char[idx] for idx in generated])
    return generated_str

def _top_k_filter(logits, k):
    values, _ = torch.topk(logits, k)
    min_values = values[:, -1].unsqueeze(1)
    return torch.where(logits < min_values, torch.ones_like(logits)*-float('inf'), logits)

def _top_p_filter(logits, p):
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

    sorted_indices_to_remove = cumulative_probs > p
    sorted_indices_to_remove[..., 0] = 0
    indices_to_remove = sorted_indices_to_remove.scatter(
        1, sorted_indices, sorted_indices_to_remove
    )
    return logits.masked_fill(indices_to_remove, -float('inf'))

In [5]:
def train_model(model, dataset, epochs=50, batch_size=32, lr=3e-4,
                warmup_epochs=5, weight_decay=0.01,
                prompt_for_sample: str = 'In this research '):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    def warmup_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        return 1.0

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=warmup_lambda)

    writer = SummaryWriter(
        log_dir=f"runs/LR_{lr:.6f}-wd_{weight_decay}-warmup_{warmup_epochs}-model_type_{model.model_type}"
    )

    best_loss = float('inf')
    grad_norms = []
    max_grad_norm = 1.0

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        progress = tqdm(loader, desc=f"Epoch {epoch+1}", leave=False)

        for batch_idx, (inputs, targets) in enumerate(progress):
            current_batch_size = inputs.size(0)
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            optimizer.zero_grad()

            hidden = model.init_hidden(current_batch_size, device)
            outputs, _ = model(inputs, hidden)
            loss = criterion(outputs.transpose(1, 2), targets)

            loss.backward()

            grad_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=max_grad_norm,
                norm_type=2,
                error_if_nonfinite=False
            )
            grad_norms.append(grad_norm.item())

            optimizer.step()

            total_loss += loss.item()
            progress.set_postfix({
                'loss': f"{loss.item():.4f}",
                'grad': f"{grad_norm:.2f}",
                'lr': f"{optimizer.param_groups[0]['lr']:.2e}"
            })

            if batch_idx % 10 == 0:
                writer.add_scalar('Train/Loss', loss.item(), epoch*len(loader)+batch_idx)
                writer.add_scalar('Train/Grad_Norm', grad_norm.item(), epoch*len(loader)+batch_idx)
                writer.add_scalar('LR', optimizer.param_groups[0]['lr'], epoch*len(loader)+batch_idx)

        scheduler.step()

        avg_loss = total_loss / len(loader)
        writer.add_scalar('Epoch/Loss', avg_loss, epoch)

        logging.info(f"Epoch {epoch+1}/{epochs} - "
                     f"Loss: {avg_loss:.4f} - "
                     f"Grad Norm: {grad_norm:.2f} - "
                     f"LR: {optimizer.param_groups[0]['lr']:.2e}")

        if avg_loss < best_loss and not torch.isnan(torch.tensor(avg_loss)):
            best_loss = avg_loss
            torch.save({
                'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'epoch': epoch,
                'loss': avg_loss
            }, f"best_model_{model.model_type}.pth")

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        model.eval()
        with torch.inference_mode():
            gen_sample = generate_sample(
                model, dataset, device,
                prompt=prompt_for_sample,
                max_length=150,
                temperature=0.7,
                top_k=5,
                top_p=0.95
            )
        logging.info(f'Gen sample after {epoch+1}/{epochs}: {gen_sample}')
        model.train()

    writer.close()
    return model

In [6]:
def rnn_train_interface(params, dataset):
	model = CharRNN(
		input_size=dataset.vocab_size,
		hidden_size=params['hidden_size'],
		output_size=dataset.vocab_size,
		model=params['model_type'],
		n_layers=params['n_layers'],
		dropout=params['dropout']
	)

	model = train_model(
		model,
		dataset,
		epochs=params['epochs'],
		batch_size=params['batch_size'],
		lr=params['lr'],
		prompt_for_sample = params['prompt'],
		warmup_epochs=params['warmup_epochs'],
		weight_decay=params['weight_decay']
	)

	return model

### HF Transformer

In [ ]:
class RussianGPT:
    def __init__(self, config):
        self.config = config
        self.model = None
        self.tokenizer = None
        self.dataset = None

        os.makedirs(self.config["output_dir"], exist_ok=True)
        os.makedirs(self.config["log_dir"], exist_ok=True)

    def prepare_data(self):
        logger.info("Loading and chunking data...")
        df = pd.read_json(self.config["data_path"])
        full_text = "\n".join(df["title"] + " " + df["text"])
        chunked_text = [
            full_text[i : i + self.config["chunk_size"]]
            for i in range(0, len(full_text), self.config["chunk_size"])
        ]
        self.dataset = Dataset.from_dict({"text": chunked_text})
        return self

    def split_dataset(self):
        logger.info("Splitting dataset...")
        split_dataset = self.dataset.train_test_split(
            test_size=self.config["test_size"], shuffle=True, seed=42
        )
        self.dataset = split_dataset
        return self

    def load_tokenizer(self):
        logger.info("Loading pre-trained tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.config["model_checkpoint"], use_fast=True
        )
        return self

    def initialize_model(self):
        logger.info("Loading pre-trained model...")
        self.model = AutoModelForCausalLM.from_pretrained(self.config["model_checkpoint"])
        logger.info(f"Model loaded with {self.model.num_parameters() / 1e6:.2f}M parameters")
        return self

    def tokenize_dataset(self):
        logger.info("Tokenizing dataset...")

        def tokenize_fn(examples):
            return self.tokenizer(
                examples["text"],
                truncation=True,
                max_length=self.config["n_positions"],
                padding="max_length",
                add_special_tokens=True,
            )

        self.dataset = self.dataset.map(
            tokenize_fn, batched=True, num_proc=1, remove_columns=["text"]
        )
        return self

    def train(self):
        logger.info(f"Dataset features: {self.dataset}")
        torch.cuda.empty_cache()

        training_args = TrainingArguments(
            output_dir=self.config["output_dir"],
            logging_dir=self.config["log_dir"],
            num_train_epochs=self.config["num_epochs"],
            per_device_train_batch_size=self.config["batch_size"],
            per_device_eval_batch_size=self.config["batch_size"] // 2,
            gradient_accumulation_steps=self.config["grad_accum_steps"],
            learning_rate=self.config["learning_rate"],
            weight_decay=0.1,
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=100,
            save_steps=250,
            eval_steps=250,
            eval_strategy="steps",
            optim="adamw_torch_fused",
            gradient_checkpointing=True,
            report_to=["tensorboard"],
            dataloader_num_workers=4,
            torch_compile=True,
            resume_from_checkpoint=True,
            save_total_limit=2,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            max_grad_norm=1.0,
            warmup_steps=250,
            lr_scheduler_type="cosine",
        )

        training_callback = TrainingProgressCallback()
        save_tokenizer_callback = SaveTokenizerCallback()
        inference_callback = InferenceLoggingCallback(self.config.get("eval_prompt", "Пример инференса"))

        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=self.dataset["train"],
            eval_dataset=self.dataset["test"],
            data_collator=DataCollatorForLanguageModeling(
                tokenizer=self.tokenizer, mlm=False
            ),
            callbacks=[training_callback, save_tokenizer_callback, inference_callback],
        )

        inference_callback.trainer = trainer

        logging.info(
            f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB"
        )
        logger.info("Starting training...")
        trainer.train()
        logger.info("Training completed. Saving final model...")

        self.model.save_pretrained(os.path.join(self.config["output_dir"], "final_model"))
        self.tokenizer.save_pretrained(os.path.join(self.config["output_dir"], "final_model"))
        return self


class TrainingProgressCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if state.is_local_process_zero:
            loss = logs.get("loss", float("nan"))
            eval_loss = logs.get("eval_loss", float("nan"))
            lr = logs.get("learning_rate", float("nan"))
            loss_str = f"{loss:.4f}" if not math.isnan(loss) else "N/A"
            eval_str = f"{eval_loss:.4f}" if not math.isnan(eval_loss) else "N/A"
            lr_str = f"{lr:.2e}" if not math.isnan(lr) else "N/A"
            logger.info(
                f"Step {state.global_step} | Loss: {loss_str} | Eval Loss: {eval_str} | Learning Rate: {lr_str}"
            )


class SaveTokenizerCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        trainer = kwargs.get("trainer", None)
        if trainer is not None and hasattr(trainer.data_collator, "tokenizer") and trainer.data_collator.tokenizer is not None:
            tokenizer_save_path = os.path.join(args.output_dir, f"checkpoint-{state.global_step}")
            trainer.data_collator.tokenizer.save_pretrained(tokenizer_save_path)
            logger.info(f"Tokenizer saved at {tokenizer_save_path}")
        return control


class InferenceLoggingCallback(TrainerCallback):
    """
    Callback для логирования инференс-вывода каждые eval_steps.
    Использует фиксированный prompt, задаваемый в конфигурации (ключ 'eval_prompt').
    """
    def __init__(self, eval_prompt: str):
        self.eval_prompt = eval_prompt
        self.trainer = None  # Будет установлен вручную

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if self.trainer is None:
            logger.warning("Trainer не найден в on_evaluate")
            return control

        # Используем токенизатор из data_collator, так как атрибут tokenizer уже deprecated
        if not hasattr(self.trainer.data_collator, "tokenizer") or self.trainer.data_collator.tokenizer is None:
            logger.warning("Tokenizer не найден в trainer.data_collator")
            return control

        tokenizer = self.trainer.data_collator.tokenizer
        inputs = tokenizer(self.eval_prompt, return_tensors="pt")
        inputs = {k: v.to(self.trainer.model.device) for k, v in inputs.items()}

        generated_ids = self.trainer.model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=True,
            temperature=1.0,
            top_p=0.95,
            repetition_penalty=1.0,
        )
        generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
        logger.info(
            f"Inference at step {state.global_step} | Prompt: '{self.eval_prompt}' | Generated: {generated_text}"
        )
        return control

## Ход работы

### 1. Тренировка RNN на arxiv

In [ ]:
params_1 = {
	'model_type': 'lstm',
	'hidden_size': 200,
	'n_layers': 2,
	'dropout': 0.25,
	'lr': 3e-3,
	'batch_size': 256,
	'epochs': 10,
	'prompt': 'In this research ',
    "warmup_epochs": 5,
    "weight_decay": 0.01
}

dataset = TextDataset(
    text=' '.join(pd.read_csv('data/arxiv.csv')['summary'].dropna().values),
    chunk_len=250, stride=100
)

best_model_lstm = rnn_train_interface(params_1, dataset)

In [10]:
params_2 = {
	'model_type': 'gru',
	'hidden_size': 200,
	'n_layers': 2,
	'dropout': 0.25,
	'lr': 3e-3,
	'batch_size': 256,
	'epochs': 10,
	'prompt': 'In this research ',
    "warmup_epochs": 5,
    "weight_decay": 0.01
}

best_model_gru = rnn_train_interface(params_2, dataset)

Epoch 1:   0%|          | 0/1147 [00:00<?, ?it/s]

2025-03-07 21:39:31,119 - INFO - Epoch 1/10 - Loss: 2.2254 - Grad Norm: 0.07 - LR: 1.20e-03
2025-03-07 21:39:31,226 - INFO - Gen sample after 1/10: In this research structure of a novel control approach to consider the control set of the state-of-the-art approach is the pose of the parameters of the analysis and t


Epoch 2:   0%|          | 0/1147 [00:00<?, ?it/s]

2025-03-07 21:40:22,949 - INFO - Epoch 2/10 - Loss: 1.8757 - Grad Norm: 0.09 - LR: 1.80e-03
2025-03-07 21:40:23,049 - INFO - Gen sample after 2/10: In this research and the simple structure of the
state-of-the-art methods for the complexity of the popular time sensing and the relation and the
state-of-the-art al


Epoch 3:   0%|          | 0/1147 [00:00<?, ?it/s]

...

2025-03-07 21:45:43,151 - INFO - Epoch 8/10 - Loss: 1.7818 - Grad Norm: 0.11 - LR: 3.00e-03
2025-03-07 21:45:43,265 - INFO - Gen sample after 8/10: In this research and the number of accuracy and an information that the
state-of-the-art methods for the classification of a simple sequence to a
set of classificati


Epoch 9:   0%|          | 0/1147 [00:00<?, ?it/s]

2025-03-07 21:46:36,437 - INFO - Epoch 9/10 - Loss: 1.7787 - Grad Norm: 0.08 - LR: 3.00e-03
2025-03-07 21:46:36,536 - INFO - Gen sample after 9/10: In this research as the convergence of the proposed method is an algorithm to achieve a set of applications of the computational
set of security and specific processe


Epoch 10:   0%|          | 0/1147 [00:00<?, ?it/s]

2025-03-07 21:47:30,600 - INFO - Epoch 10/10 - Loss: 1.7760 - Grad Norm: 0.08 - LR: 3.00e-03
2025-03-07 21:47:30,707 - INFO - Gen sample after 10/10: In this research specific to the same training and analysis of the popular and a single state-of-the-art methods and the computational patterns of the analysis of a
s


### 2. Тренировка RNN на РИА Новости

In [40]:
df = pd.read_json('data/ria_articles.json')['text']

In [43]:
def clean_text(text):
    text = re.sub(r'^.*?РИА Новости(?:, [^.\n]+)?\.\s*', '', text)
    return '.'.join(text.split('.', 2)[:2]) if text.count('.') >= 2 else text

df = df.apply(clean_text)

In [44]:
json_output = df.to_json('data/ria_new.json', force_ascii=False, orient='records', indent=4)

In [ ]:
dataset_ria = TextDataset(
    text=' '.join(df.dropna().values),
    chunk_len=250, stride=100
)

In [46]:
params_1_ria = {
	'model_type': 'lstm',
	'hidden_size': 200,
	'n_layers': 2,
	'dropout': 0.25,
	'lr': 3e-3,
	'batch_size': 256,
	'epochs': 10,
	'prompt': 'Россия будет ',
    "warmup_epochs": 5,
    "weight_decay": 0.01
}

best_model_lstm_ria = rnn_train_interface(params_1_ria, dataset_ria)

Epoch 1:   0%|          | 0/1111 [00:00<?, ?it/s]

2025-03-07 23:13:03,845 - INFO - Epoch 1/10 - Loss: 2.9536 - Grad Norm: 0.11 - LR: 1.20e-03
2025-03-07 23:13:03,951 - INFO - Gen sample after 1/10: Россия будет проведения соболичной представления в расстоком в представитель в области с проведение страны в развития с президент президент России в протодне в про


Epoch 2:   0%|          | 0/1111 [00:00<?, ?it/s]

2025-03-07 23:14:01,202 - INFO - Epoch 2/10 - Loss: 2.3610 - Grad Norm: 0.10 - LR: 1.80e-03
2025-03-07 23:14:01,337 - INFO - Gen sample after 2/10: Россия будет против на последние с получения на президента РФ в соцсети по объекта подразделения в пресс-службе сообщили в политика по собрания с возможность в пре


Epoch 3:   0%|          | 0/1111 [00:00<?, ?it/s]

...

2025-03-07 23:19:46,203 - INFO - Epoch 8/10 - Loss: 1.9944 - Grad Norm: 0.07 - LR: 3.00e-03
2025-03-07 23:19:46,307 - INFO - Gen sample after 8/10: Россия будет продолжить в развитии военного выпуска в составе проверки состояния стали недавно под собственными использованиями в стране страны в столице и при все


Epoch 9:   0%|          | 0/1111 [00:00<?, ?it/s]

2025-03-07 23:20:43,725 - INFO - Epoch 9/10 - Loss: 1.9849 - Grad Norm: 0.07 - LR: 3.00e-03
2025-03-07 23:20:43,830 - INFO - Gen sample after 9/10: Россия будет полностью с возможности при отражении продолжать подразделения группировки войск "Запад" провели состав в районе Владимира Зеленского с проведением со


Epoch 10:   0%|          | 0/1111 [00:00<?, ?it/s]

2025-03-07 23:21:41,572 - INFO - Epoch 10/10 - Loss: 1.9777 - Grad Norm: 0.08 - LR: 3.00e-03
2025-03-07 23:21:41,673 - INFO - Gen sample after 10/10: Россия будет воздушный суд по вопросам состояния на пострадавший на выборах президента РФ на выборах главы региона и полученного выбора о получении совершения подд


In [47]:
params_2_ria = {
	'model_type': 'gru',
	'hidden_size': 200,
	'n_layers': 2,
	'dropout': 0.25,
	'lr': 3e-3,
	'batch_size': 256,
	'epochs': 10,
	'prompt': 'Россия будет ',
    "warmup_epochs": 5,
    "weight_decay": 0.01
}

best_model_gru_ria = rnn_train_interface(params_2_ria, dataset_ria)

Epoch 1:   0%|          | 0/1111 [00:00<?, ?it/s]

2025-03-07 23:22:34,673 - INFO - Epoch 1/10 - Loss: 2.6460 - Grad Norm: 0.08 - LR: 1.20e-03
2025-03-07 23:22:34,775 - INFO - Gen sample after 1/10: Россия будет Калининской области с поддержку противника в последние своей предложении с президента РФ в Москве в пресс-службе совершения на протоверки на проведени


Epoch 2:   0%|          | 0/1111 [00:00<?, ?it/s]

2025-03-07 23:23:28,778 - INFO - Epoch 2/10 - Loss: 2.1949 - Grad Norm: 0.10 - LR: 1.80e-03
2025-03-07 23:23:28,883 - INFO - Gen sample after 2/10: Россия будет Воздушной представитель из дома на поле в России с отношении правительства РФ Владимира Путина по следующем проведении контроля и составляет президент


Epoch 3:   0%|          | 0/1111 [00:00<?, ?it/s]

...

2025-03-07 23:28:53,868 - INFO - Epoch 8/10 - Loss: 2.0514 - Grad Norm: 0.11 - LR: 3.00e-03
2025-03-07 23:28:53,978 - INFO - Gen sample after 8/10: Россия будет принимать возможность представителей в России пострадали объявлен на сайте Картона и после события для продажи возможности страны после обработки 15 м


Epoch 9:   0%|          | 0/1111 [00:00<?, ?it/s]

2025-03-07 23:29:48,325 - INFO - Epoch 9/10 - Loss: 2.0461 - Grad Norm: 0.11 - LR: 3.00e-03
2025-03-07 23:29:48,428 - INFO - Gen sample after 9/10: Россия будет сообщила пресс-служба правительства РФ Владимир Путин поручил председатель МИД РФ Дмитрий Песков.
"Подразделения группировки войск "Центр" с применени


Epoch 10:   0%|          | 0/1111 [00:00<?, ?it/s]

2025-03-07 23:30:42,894 - INFO - Epoch 10/10 - Loss: 2.0420 - Grad Norm: 0.10 - LR: 3.00e-03
2025-03-07 23:30:42,989 - INFO - Gen sample after 10/10: Россия будет против российской кампании с передаче в составе концертного собора на переговорах с президентом России Владимира Путина в развитии сериала "Север" в р


### 3. Fine-Tuning Transformer на РИА Новости

In [15]:
def inference_transformer(
    prompt: str,
    model_dir: str,
    n: int = 1,
    repetition_penalty: float = 1.0,
    temperature: float = 1.0,
    top_p: float = 1.0,
    top_k: int = -1,
    max_tokens: int = 50,
    min_tokens: int = 0,
    skip_special_tokens: bool = True,
):
    # Загружаем токенайзер и модель из указанной директории
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForCausalLM.from_pretrained(model_dir)

    # Переводим модель на нужное устройство (GPU, если доступен)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Токенизируем входной текст
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {key: val.to(device) for key, val in inputs.items()}

    # Если top_k равен -1, отключаем top-k фильтрацию
    if top_k <= 0:
        top_k = None

    # Вычисляем параметры генерации.
    # Используем max_new_tokens и min_new_tokens, чтобы задать число генерируемых токенов
    do_sample = temperature > 0.0  # при temperature == 0 происходит жадное декодирование

    generate_kwargs = {
        "do_sample": do_sample,
        "num_return_sequences": n,
        "repetition_penalty": repetition_penalty,
        "top_p": top_p,
        "max_new_tokens": max_tokens,
        "min_new_tokens": min_tokens,
        "eos_token_id": tokenizer.eos_token_id,
    }
    if do_sample:
        generate_kwargs["temperature"] = temperature
    if top_k is not None:
        generate_kwargs["top_k"] = top_k

    # Генерация последовательностей
    outputs = model.generate(**inputs, **generate_kwargs)

    # Декодируем результаты
    responses = [
        tokenizer.decode(output, skip_special_tokens=skip_special_tokens)
        for output in outputs
    ]
    return responses

In [ ]:
config = {
    "data_path": "data/ria_articles.json",
    "test_size": 0.05,
    "output_dir": f"./models/rugpt-{datetime.now().strftime('%Y%m%d-%H%M')}",
    "log_dir": "./logs",
    "chunk_size": 1024,
    "n_positions": 1024,
    "num_epochs": 5,
    "batch_size": 16,
    "grad_accum_steps": 4,
    "learning_rate": 5e-4,
    "eval_prompt": "Путин сказал: ",
    "model_checkpoint": "sberbank-ai/rugpt3small_based_on_gpt2",
}

In [ ]:
(
	RussianGPT(config)
	.prepare_data()
	.load_tokenizer()
	.split_dataset()
	.initialize_model()
	.tokenize_dataset()
	.train()
)

In [31]:
def inference_rnn(
        model_path: str, model_params: dict, dataset: TextDataset,
        prompt: str,
        max_length: int,
        temperature: float,
        top_k: int,
        top_p: float
    ):
    model = CharRNN(**model_params).to('cuda')
    model.load_state_dict(torch.load(model_path, weights_only=True)['model'])
    sample = generate_sample(
        model, dataset, torch.device('cuda'), 
        prompt=prompt,
        max_length=max_length,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p
    )
    return sample

In [ ]:
from copy import deepcopy

ENG_PROMPT = "The main aim in this research is "
RU_PROMPT = "Путин сказал, что "

RU_RNN_PARAMS_BASE = {
    "input_size": 241,
    "hidden_size": 200,
    "output_size": 241,
    'n_layers': 2,
    'dropout': 0.25,
    'model': None
}
lstm_params_ru = deepcopy(RU_RNN_PARAMS_BASE)
lstm_params_ru['model'] = 'lstm'
gru_params_ru = deepcopy(RU_RNN_PARAMS_BASE)
gru_params_ru['model'] = 'gru'

ENG_RNN_PARAMS_BASE = {
    "input_size": 98,
    "hidden_size": 200,
    "output_size": 98,
    'n_layers': 2,
    'dropout': 0.25,
    'model': None
}
lstm_params_eng = deepcopy(ENG_RNN_PARAMS_BASE)
lstm_params_eng['model'] = 'lstm'
gru_params_eng = deepcopy(ENG_RNN_PARAMS_BASE)
gru_params_eng['model'] = 'gru'

dataset_arxiv = TextDataset(
    text=' '.join(pd.read_csv('data/arxiv.csv')['summary'].dropna().values),
    chunk_len=250, stride=100
)


df = pd.read_json('data/ria_articles_concatenated.json')['text']
def clean_text(text):
    text = re.sub(r'^.*?РИА Новости(?:, [^.\n]+)?\.\s*', '', text)
    return '.'.join(text.split('.', 2)[:2]) if text.count('.') >= 2 else text

df = df.apply(clean_text)

dataset_ria = TextDataset(
    text=' '.join(df.dropna().values),
    chunk_len=250, stride=100
)

temp_list = [0.1, 0.7, 1.4]
top_k_list = [5, 20, 40]
top_p_list = [0.5, 0.7, 0.95]

In [60]:
import itertools

for temp, top_k, top_p in itertools.product(temp_list, top_k_list, top_p_list):
    response_lstm_arixv = inference_rnn(
        model_path='pretrained/arxiv/best_model_lstm.pth',
        model_params=lstm_params_eng,
        dataset=dataset_arxiv,
        prompt=ENG_PROMPT,
        max_length=100,
        temperature=temp,
        top_k=top_k,
        top_p=top_p
    )

    response_gru_arixv = inference_rnn(
        model_path='pretrained/arxiv/best_model_gru.pth',
        model_params=gru_params_eng,
        dataset=dataset_arxiv,
        prompt=ENG_PROMPT,
        max_length=100,
        temperature=temp,
        top_k=top_k,
        top_p=top_p
    )

    print('-' * 50)
    print(f"* Temperature: {temp}\n*Top_k: {top_k}\n*Top_p {top_p}")
    print('-' * 50)
    print(f'Arxiv LSTM output:\n{response_lstm_arixv}\n')
    print(f'Arxiv GRU output:\n{response_gru_arixv}\n\n')

--------------------------------------------------
* Temperature: 0.1
*Top_k: 5
*Top_p 0.5
--------------------------------------------------
Arxiv LSTM output:
The main aim in this research is a set of sets of the state of the art and the state-of-the-art algorithms to achieve the state-of-th

Arxiv GRU output:
The main aim in this research is as the proposed approach is the context of the proposed approach is the context of the proposed appr


--------------------------------------------------
* Temperature: 0.1
*Top_k: 5
*Top_p 0.7
--------------------------------------------------
Arxiv LSTM output:
The main aim in this research is a set of sets of the state of the art and the state-of-the-art algorithms to achieve the state-of-th

Arxiv GRU output:
The main aim in this research is as the proposed approach is the context of the proposed approach is the context of the proposed appr


--------------------------------------------------
* Temperature: 0.1
*Top_k: 5
*Top_p 0.95
--------

In [62]:
for temp, top_k, top_p in itertools.product(temp_list, top_k_list, top_p_list):
    response_lstm_ria = inference_rnn(
        model_path='pretrained/ria/best_model_lstm.pth',
        model_params=lstm_params_ru,
        dataset=dataset_ria,
        prompt=RU_PROMPT,
        max_length=100,
        temperature=temp,
        top_k=top_k,
        top_p=top_p
    )

    response_gru_ria = inference_rnn(
        model_path='pretrained/ria/best_model_gru.pth',
        model_params=gru_params_ru,
        dataset=dataset_ria,
        prompt=RU_PROMPT,
        max_length=100,
        temperature=temp,
        top_k=top_k,
        top_p=top_p
    )

    print('-' * 50)
    print(f"* Temperature: {temp}\n*Top_k: {top_k}\n*Top_p {top_p}")
    print('-' * 50)
    print(f'RIA LSTM output:\n{response_lstm_ria}\n')
    print(f'RIA GRU output:\n{response_gru_ria}\n\n')

--------------------------------------------------
* Temperature: 0.1
*Top_k: 5
*Top_p 0.5
--------------------------------------------------
RIA LSTM output:
Путин сказал, что в составе продолжают продолжать продолжить продолжительное продолжительное продолжительное продолжит

RIA GRU output:
Путин сказал, что на политическом пространстве в соцсети X .
"В стране в соцсети X .
"В стране в соцсети X .
"В стране


--------------------------------------------------
* Temperature: 0.1
*Top_k: 5
*Top_p 0.7
--------------------------------------------------
RIA LSTM output:
Путин сказал, что в составе продолжают продолжать продолжить продолжительное продолжительное продолжительное продолжит

RIA GRU output:
Путин сказал, что на политическом пространстве в соцсети X .
"В стране в соцсети X .
"В стране в соцсети X .
"В стране


--------------------------------------------------
* Temperature: 0.1
*Top_k: 5
*Top_p 0.95
--------------------------------------------------
RIA LSTM output:
Путин ск

In [63]:
repetition_penalty_list = [0.5, 1.0, 1.5]

for repetition_penalty, temperature, top_k, top_p in itertools.product(repetition_penalty_list, temp_list, top_k_list, top_p_list):
    responses = inference_transformer(
        prompt=RU_PROMPT,
        model_dir='models/rugpt-20250305-1910/final_model',
        n=1,
        repetition_penalty=repetition_penalty,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        max_tokens=30,
        min_tokens=15,
        skip_special_tokens=True
    )

    print('-' * 50)
    print(f"Repetition Penalty: {repetition_penalty}\n* Temperature: {temp}\n*Top_k: {top_k}\n*Top_p {top_p}")
    print('-' * 50)
    print(responses[0] + '\n')

--------------------------------------------------
Repetition Penalty: 0.5
* Temperature: 1.4
*Top_k: 5
*Top_p 0.5
--------------------------------------------------
Путин сказал, что  сказал, что  сказал, что  сказал, что  сказал, что  сказал, что  сказал, что  сказал, что  сказал,

--------------------------------------------------
Repetition Penalty: 0.5
* Temperature: 1.4
*Top_k: 5
*Top_p 0.7
--------------------------------------------------
Путин сказал, что  сказал, что  сказал, что  сказал, что  сказал, что  сказал, что  сказал, что  сказал, что  сказал,

--------------------------------------------------
Repetition Penalty: 0.5
* Temperature: 1.4
*Top_k: 5
*Top_p 0.95
--------------------------------------------------
Путин сказал, что  сказал, что  сказал, что  сказал, что  сказал, что  сказал, что  сказал, что  сказал, что  сказал,

--------------------------------------------------
Repetition Penalty: 0.5
* Temperature: 1.4
*Top_k: 20
*Top_p 0.5
----------------------------

### Выводы по ходу работы